In [1]:
import os
import gymnasium
import highway_env
import warnings
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

env_name = 'highway-v0'
root = f'{env_name}-SAC'

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── Configuración ────────────────────────────────────────────────────────────
config = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 15,
        "features": ["presence", "x", "y", "vx", "vy"],
        "features_range": {
            "x": [-100, 100],
            "y": [-100, 100],
            "vx": [-30, 30],
            "vy": [-30, 30],
        },
        "absolute": False,
        "order": "sorted",
        "normalize": True,
    },

    "action": {
        "type": "ContinuousAction",
    },

    "lanes_count": 4,
    "vehicles_count": 50,
    "vehicles_density": 1.2,
    "duration": 60,
    "initial_lane_id": None,

    "simulation_frequency": 15,
    "policy_frequency": 5,

    "collision_reward": -2.0,
    "high_speed_reward": 2.5,
    "lane_change_reward": -0.05,
    "right_lane_reward": 0.0,
    "reward_speed_range": [20, 30],
    "normalize_reward": True,

    "offroad_terminal": True,
    "controlled_vehicles": 1,
    "manual_control": False,
}

log_dir = f"{root}/logs/"
os.makedirs(log_dir, exist_ok=True)

# ── Entornos ─────────────────────────────────────────────────────────────────
def make_env():
    env = gymnasium.make(env_name)
    env.unwrapped.configure(config)
    env.reset()
    env = Monitor(env, log_dir)
    return env

env = make_env()

# ── Modelo (Carga o Creación) ──────────────────────────────────────────────────
TOTAL_TIMESTEPS = 100_000

print("Creando modelo desde cero...")
model = SAC(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    train_freq=1,
    gradient_steps=1,
    ent_coef="auto",
    verbose=1,
)

# ── Entrenamiento ─────────────────────────────────────────────────────────────

try:
    print("Entrenando al agente...")
    model.learn(total_timesteps=TOTAL_TIMESTEPS)
except KeyboardInterrupt:
    print("\nEntrenamiento interrupido por el usuario (Ctrl+C). Guardando estado actual...")

finally:
    print("Guardando modelo...")
    model.save(f"{log_dir}/sac_model")

    env.close()
    print("Entornos cerrados correctamente.")

<frozen importlib._bootstrap>:488: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


Creando modelo desde cero...
Using cpu device
Wrapping the env in a DummyVecEnv.
Entrenando al agente...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 2.25     |
|    ep_rew_mean     | 0.673    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 9        |
|    time_elapsed    | 0        |
|    total_timesteps | 9        |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.12     |
|    ep_rew_mean     | 1.23     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 10       |
|    time_elapsed    | 2        |
|    total_timesteps | 25       |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.5      |
|    ep_rew_mean     | 1.44     |
| time/              |          |
|    episodes        | 12       |
|    fps   